In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

INPUT_FILE  = "cleaning_data.csv"
OUTPUT_FILE = "cleaned_data.csv"

def is_valid_email(val):
    pattern = r"^[\w\.\+\-]+@[\w\-]+\.[a-zA-Z]{2,}$"
    return bool(re.match(pattern, str(val))) if pd.notna(val) else False

def parse_date_flexible(val):
    if pd.isna(val) or str(val).strip().lower() in ("not available", "na", "n/a", ""):
        return pd.NaT
    for fmt in ("%Y/%m/%d", "%d/%m/%Y", "%m-%d-%Y", "%Y-%m-%d",
                "%d-%m-%Y", "%m/%d/%Y", "%Y%m%d"):
        try:
            return pd.to_datetime(str(val).strip(), format=fmt)
        except ValueError:
            continue
    try:
        return pd.to_datetime(str(val).strip(), infer_datetime_format=True)
    except Exception:
        return pd.NaT

COUNTRY_MAP = {
    "deutschland": "Germany",
    "germany":     "Germany",
    "usa":         "United States",
    "us":          "United States",
    "united states of america": "United States",
    "uk":          "United Kingdom",
    "great britain": "United Kingdom",
}

df_raw = pd.read_csv(INPUT_FILE)
print(f"Loaded '{INPUT_FILE}': {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head(5)

Loaded 'cleaning_data.csv': 100 rows x 18 columns


,Order_ID,Customer_ID,Customer_Name,Email,Country,City,Order_Date,Ship_Date,Product_Category,Quantity,Unit_Price,Discount,Sales,Age,Customer_Segment,Satisfaction_Score,Payment_Method,Notes
0,O1001,C016,MAHA ZADEH,maha.zadeh@example.com,Deutschland,Mannheim,2026/01/27,not available,office supplies,2,19.73,0.25,29.59,60.0,Corporate,5.0,Bank Transfer,ok
1,O1002,C025,Sara Smith,sara.smith@example.com,germany,karlsruhe,03-30-2026,2026-04-08,electronics,7,285.86,0.10,1800.92,52.0,CORPORATE,NaN,Bank Transfer,ok
2,O1003,C014,Carlos Garcia,carlos.garcia@example.com,Italy,Rome,19/01/2026,2026/01/28,Office Supplies,7,312.17,0.10,1966.67,69.0,consumer,1.0,NaN,check address
3,O1004,C009,Nina Weber,nina.weber@example.com,Deutschland,Mannheim,2026/05/18,22.05.2026,Home,4,226.13,0.00,904.52,25.0,Corporate,4.0,Credit Card,return risk
4,O1005,C018,Maha Zadeh,maha.zadeh@example.com,italy,Naples,29.01.2026,02/02/2026,Electronics,12,86.96,0.25,782.64,35.0,Home Office,2.0,bank transfer,check address


## Step 2 — Data Profiling

In [ ]:
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns\n")
print("Columns:", df_raw.columns.tolist())

Shape: 100 rows x 18 columns

Column Names:
['Order_ID', 'Customer_ID', 'Customer_Name', 'Email', 'Country', 'City', 'Order_Date', 'Ship_Date', 'Product_Category', 'Quantity', 'Unit_Price', 'Discount', 'Sales', 'Age', 'Customer_Segment', 'Satisfaction_Score', 'Payment_Method', 'Notes']


In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
miss_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
miss_df = miss_df[miss_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)
print("Missing Values:")
display(miss_df)

Missing Values:


,Missing Count,Missing %
Satisfaction_Score,15,15.0
Payment_Method,14,14.0
Customer_Segment,12,12.0
Email,10,10.0
Notes,10,10.0
Age,7,7.0
Order_Date,5,5.0
City,3,3.0


In [ ]:
print("Data Types:")
display(df_raw.dtypes.rename("dtype").to_frame())
print("\nNote: Unit_Price is object — needs numeric conversion")

Data Types:


,dtype
Order_ID,object
Customer_ID,object
Customer_Name,object
Email,object
Country,object
City,object
Order_Date,object
Ship_Date,object
Product_Category,object
Quantity,int64



Note: Unit_Price is object (string) -- needs numeric conversion


In [ ]:
print("Numeric Summary:")
display(df_raw.select_dtypes(include="number").describe().round(2))
print("\nNotes:")
print("  Quantity min=0, Age max=130 — likely invalid values")
print("  Sales std is very high (7144) vs mean (2477) — possible outliers")

Numeric Summary Statistics:


,Quantity,Discount,Sales,Age,Satisfaction_Score
count,100.00,100.00,100.00,93.00,85.00
mean,5.77,0.13,2477.78,48.49,3.12
std,3.34,0.08,7144.01,19.67,1.25
min,0.00,0.00,0.00,16.00,1.00
25%,3.00,0.05,376.58,35.00,2.00
50%,6.00,0.15,1246.68,46.00,3.00
75%,8.00,0.20,2012.24,60.00,4.00
max,12.00,0.25,56999.94,130.00,5.00



Observations:
  - Quantity min=0, Age max=130 --> likely invalid values
  - Sales std is very high (7144) vs mean (2477) --> outliers


In [ ]:
dup_count = df_raw.duplicated().sum()
print(f"Fully duplicate rows: {dup_count}")
if dup_count > 0:
    print("\nSample duplicates:")
    display(df_raw[df_raw.duplicated(keep=False)].sort_values("Order_ID").head(6))

Fully duplicate rows: 5

Sample duplicate rows:


,Order_ID,Customer_ID,Customer_Name,Email,Country,City,Order_Date,Ship_Date,Product_Category,Quantity,Unit_Price,Discount,Sales,Age,Customer_Segment,Satisfaction_Score,Payment_Method,Notes
4,O1005,C018,Maha Zadeh,maha.zadeh@example.com,italy,Naples,29.01.2026,02/02/2026,Electronics,12,86.96,0.25,782.64,35.0,Home Office,2.0,bank transfer,check address
95,O1005,C018,Maha Zadeh,maha.zadeh@example.com,italy,Naples,29.01.2026,02/02/2026,Electronics,12,86.96,0.25,782.64,35.0,Home Office,2.0,bank transfer,check address
17,O1018,C049,MEHDI KARIMI,mehdi.karimi@example.com,Netherlands,Amsterdam,28/01/2026,03/02/2026,Home,8,382.96,0.15,2604.13,58.0,CORPORATE,3.0,NaN,missing phone
96,O1018,C049,MEHDI KARIMI,mehdi.karimi@example.com,Netherlands,Amsterdam,28/01/2026,03/02/2026,Home,8,382.96,0.15,2604.13,58.0,CORPORATE,3.0,NaN,missing phone
28,O1029,C010,NINA WEBER,nina.weber@example.com,NETHERLANDS,Rotterdam,2026-02-06,07/02/2026,Office Supplies,10,€215.16,0.15,1828.86,76.0,Home Office,4.0,Bank Transfer,return risk
97,O1029,C010,NINA WEBER,nina.weber@example.com,NETHERLANDS,Rotterdam,2026-02-06,07/02/2026,Office Supplies,10,€215.16,0.15,1828.86,76.0,Home Office,4.0,Bank Transfer,return risk


In [ ]:
rows = len(df_raw)
card = pd.DataFrame({
    "Unique Values": df_raw.nunique(),
    "Unique %": (df_raw.nunique() / rows * 100).round(1)
}).sort_values("Unique Values", ascending=False)
print("Cardinality:")
display(card)

Column Cardinality:


,Unique Values,Unique %
Order_ID,95,95.0
Unit_Price,93,93.0
Sales,92,92.0
Ship_Date,87,87.0
Order_Date,82,82.0
Customer_ID,46,46.0
Age,43,43.0
Customer_Name,39,39.0
City,15,15.0
Email,15,15.0


In [ ]:
print("Text Format Issues:\n")
for col in df_raw.select_dtypes(include="object").columns:
    vals = df_raw[col].dropna().astype(str)
    sp = (vals.str.startswith(" ") | vals.str.endswith(" ")).any()
    mixed = (vals.str.upper() != vals).any() and (vals.str.lower() != vals).any()
    uniq = df_raw[col].nunique()
    issues = []
    if sp:    issues.append("leading/trailing spaces")
    if mixed: issues.append("mixed case")
    status = ", ".join(issues) if issues else "OK"
    print(f"  {col:25s}  unique={uniq:4d}  | {status}")

print("\nDate format samples:")
print("  Order_Date:", df_raw["Order_Date"].dropna().unique()[:6].tolist())
print("  Ship_Date: ", df_raw["Ship_Date"].dropna().unique()[:6].tolist())

Text Format Issues:

  Order_ID                   unique=  95  | OK
  Customer_ID                unique=  46  | OK
  Customer_Name              unique=  39  | leading/trailing spaces, mixed case
  Email                      unique=  15  | OK
  Country                    unique=  15  | mixed case
  City                       unique=  15  | mixed case
  Order_Date                 unique=  82  | OK
  Ship_Date                  unique=  87  | OK
  Product_Category           unique=  10  | mixed case
  Unit_Price                 unique=  93  | OK
  Customer_Segment           unique=   6  | mixed case
  Payment_Method             unique=   7  | mixed case
  Notes                      unique=   5  | OK

Date format samples:
  Order_Date: ['2026/01/27', '03-30-2026', '19/01/2026', '2026/05/18', '29.01.2026', '2026/01/15']
  Ship_Date:  ['not available', '2026-04-08', '2026/01/28', '22.05.2026', '02/02/2026', '2026-01-18']


## Step 3 — Column Analysis

In [ ]:
rows = len(df_raw)
print(f"{'Column':<25} {'Unique':>8} {'Unique%':>9}  Role\n" + "-"*65)
for col in df_raw.columns:
    u = df_raw[col].nunique()
    pct = u / rows * 100
    if pct == 100:
        role = "PRIMARY KEY candidate"
    elif pct > 80:
        role = "High-cardinality (possible ID)"
    elif u <= 2:
        role = "Binary / Flag"
    elif u <= 15:
        role = "Categorical"
    else:
        role = "Continuous / Semi-unique"
    print(f"  {col:<23} {u:>8} {pct:>8.1f}%  {role}")

Column                      Unique   Unique%  Role
-----------------------------------------------------------------
  Order_ID                      95     95.0%  High-cardinality (possible ID)
  Customer_ID                   46     46.0%  Continuous / Semi-unique
  Customer_Name                 39     39.0%  Continuous / Semi-unique
  Email                         15     15.0%  Categorical
  Country                       15     15.0%  Categorical
  City                          15     15.0%  Categorical
  Order_Date                    82     82.0%  High-cardinality (possible ID)
  Ship_Date                     87     87.0%  High-cardinality (possible ID)
  Product_Category              10     10.0%  Categorical
  Quantity                      13     13.0%  Categorical
  Unit_Price                    93     93.0%  High-cardinality (possible ID)
  Discount                       6      6.0%  Categorical
  Sales                         92     92.0%  High-cardinality (possible ID)
  Age   

In [ ]:
total = df_raw["Email"].dropna().shape[0]
valid = df_raw["Email"].dropna().apply(is_valid_email).sum()
invalid = total - valid
print(f"Email: {valid}/{total} valid, {invalid} invalid")

invalid_emails = df_raw[~df_raw["Email"].apply(is_valid_email) & df_raw["Email"].notna()]["Email"]
if not invalid_emails.empty:
    print("\nInvalid email values:")
    print(invalid_emails.unique().tolist())

Email column: 90/90 valid, 0 invalid


In [ ]:
for col in ["Order_Date", "Ship_Date"]:
    bad_mask = df_raw[col].astype(str).str.lower().str.contains(
        r"not available|^na$|^n/a$", regex=True, na=False)
    print(f"{col}:")
    print(f"  Nulls: {df_raw[col].isnull().sum()}")
    print(f"  'not available': {bad_mask.sum()}")
    print(f"  Samples: {df_raw[col].dropna().unique()[:5].tolist()}\n")

Order_Date:
  Null values   : 5
  'not available': 0
  Sample values : ['2026/01/27', '03-30-2026', '19/01/2026', '2026/05/18', '29.01.2026']

Ship_Date:
  Null values   : 0
  'not available': 2
  Sample values : ['not available', '2026-04-08', '2026/01/28', '22.05.2026', '02/02/2026']



In [ ]:
print("Product_Category values:")
print(sorted(df_raw["Product_Category"].dropna().str.lower().unique()))

print("\nCustomer_Segment values:")
print(sorted(df_raw["Customer_Segment"].dropna().unique()))

print("\nCountry values:")
print(sorted(df_raw["Country"].dropna().unique()))

print("\nNumeric checks:")
for col in ["Quantity", "Discount", "Age", "Satisfaction_Score"]:
    neg  = (df_raw[col] < 0).sum() if pd.api.types.is_numeric_dtype(df_raw[col]) else "N/A"
    zero = (df_raw[col] == 0).sum() if pd.api.types.is_numeric_dtype(df_raw[col]) else "N/A"
    print(f"  {col}: negatives={neg}, zeros={zero}")

Product_Category unique values:
['electronics', 'furniture', 'home', 'office supplies']

Customer_Segment unique values:
['CORPORATE', 'Consumer', 'Corporate', 'Home Office', 'consumer', 'home office']

Country unique values:
['Deutschland', 'España', 'FRANCE', 'France', 'GERMANY', 'Germany', 'ITALY', 'Italy', 'NETHERLANDS', 'Netherlands', 'Spain', 'france', 'germany', 'italy', 'spain']

Numeric validity:
  Quantity: negatives=0, zeros=4
  Discount: negatives=0, zeros=9
  Age: negatives=0, zeros=0
  Satisfaction_Score: negatives=0, zeros=0


## Step 4 — Structure Analysis

In [ ]:
print("ID Uniqueness")
for col in ["Order_ID", "Customer_ID"]:
    u = df_raw[col].nunique()
    n = len(df_raw)
    msg = "unique — valid primary key" if u == n else f"NOT unique ({n - u} duplicates)"
    print(f"  {col}: {u}/{n} unique  -> {msg}")

print(f"\n  Distinct customers : {df_raw['Customer_ID'].nunique()}")
print(f"  Avg orders/customer: {len(df_raw) / df_raw['Customer_ID'].nunique():.1f}")

=== ID Uniqueness ===
  Order_ID: 95/100 unique  --> NOT unique (5 duplicates)
  Customer_ID: 46/100 unique  --> NOT unique (54 duplicates)

  Distinct customers : 46
  Avg orders/customer: 2.2


In [ ]:
print("Customer_ID -> Customer_Name Consistency")
name_per_cust = df_raw.groupby("Customer_ID")["Customer_Name"].nunique()
inconsistent  = name_per_cust[name_per_cust > 1]
print(f"  Customer IDs with multiple names: {len(inconsistent)}")
if not inconsistent.empty:
    print("\n  Affected customers:")
    for cid in inconsistent.index:
        names = df_raw[df_raw["Customer_ID"] == cid]["Customer_Name"].unique()
        print(f"    {cid}: {names.tolist()}")

=== Customer_ID -> Customer_Name Consistency ===
  Customer IDs with multiple names: 31

  Affected customers:
    C003: ['Luca Bianchi', 'Laura Rossi']
    C004: ['Tom Brown', 'marie dubois']
    C005: ['Maha Zadeh', 'Anna Müller', 'nina weber']
    C006: ['MARIE DUBOIS', 'luca bianchi']
    C009: ['Nina Weber', 'Luca Bianchi']
    C010: ['NINA WEBER', 'Marie Dubois']
    C012: ['Anna Müller', 'Ali Reza']
    C014: ['Carlos Garcia', 'Marie Dubois', 'nina weber']
    C017: ['Emma Wilson', 'carlos garcia']
    C018: ['Maha Zadeh', 'CARLOS GARCIA']
    C019: ['John Doe', 'Ali Reza', 'TOM BROWN', 'marie dubois', 'Julia Fischer']
    C020: ['Maha Zadeh', '  Mehdi Karimi  ']
    C021: ['mehdi karimi', 'tom brown']
    C022: ['carlos garcia', 'Anna Müller', 'John Doe']
    C024: ['John Doe', 'mehdi karimi']
    C029: ['John Doe', 'Anna Müller', 'Julia Fischer', 'JOHN DOE']
    C030: ['laura rossi', 'Nina Weber']
    C032: ['John Doe', 'Mehdi Karimi', 'Carlos Garcia']
    C034: ['Nina Weber',

In [ ]:
print("Sales Consistency Check")
df_check = df_raw.copy()
df_check["Unit_Price_num"] = pd.to_numeric(df_check["Unit_Price"], errors="coerce")
df_check["Sales_expected"] = (
    df_check["Quantity"] * df_check["Unit_Price_num"] * (1 - df_check["Discount"])
).round(2)
df_check["Sales_diff"] = (df_check["Sales"] - df_check["Sales_expected"]).abs()

mismatch = df_check[df_check["Sales_diff"] > 0.05].copy()
print(f"  Total rows: {len(df_raw)}")
print(f"  Mismatched rows: {len(mismatch)}  (tolerance = 0.05)")

if not mismatch.empty:
    print("\n  Sample mismatches:")
    display(mismatch[["Order_ID","Quantity","Unit_Price_num","Discount",
                       "Sales","Sales_expected","Sales_diff"]].head(5))

=== Calculated Field: Sales Consistency ===
  Total rows          : 100
  Rows with mismatch  : 0  (tolerance = 0.05)


## Step 5 — Cleaning Rules

In [ ]:
rules = {
    "Unit_Price (stored as text)":           "Convert to float; unparseable -> NaN -> fill with median",
    "Order_Date / Ship_Date (mixed formats)": "Parse all date formats; 'not available' -> NaT -> fill with median date",
    "Country (Deutschland, germany)":         "Normalize via lookup map -> Title Case",
    "Customer_Segment (mixed case)":          "Apply Title Case",
    "Product_Category (mixed case)":          "Apply Title Case",
    "Customer_Name (mixed case + spaces)":    "Strip whitespace, apply Title Case",
    "Email (invalid format)":                 "Replace with 'invalid@unknown.com'",
    "Email (missing)":                        "Fill with 'no-email@unknown.com'",
    "Satisfaction_Score (15% missing)":       "Fill with median",
    "Payment_Method (14% missing)":           "Fill with 'Unknown'",
    "Notes (10% missing)":                    "Fill with 'None'",
    "Age (7% missing)":                       "Fill with median",
    "Duplicate rows (5 found)":               "Drop exact duplicates, keep first",
    "Sales (formula mismatch)":               "Recalculate: Quantity x Unit_Price x (1 - Discount)",
    "Derived: Age_Group":                     "Bin Age into: <18, 18-25, 26-35, 36-50, 51-65, 65+",
    "Scaling":                                "Min-Max scale: Quantity, Unit_Price, Discount, Age, Satisfaction_Score",
}

print(f"{'Issue / Field':<45}  Cleaning Action")
print("-" * 100)
for field, action in rules.items():
    print(f"  {field:<43}  {action}")

Issue / Field                                  Cleaning Action
----------------------------------------------------------------------------------------------------
  Unit_Price (stored as text)                  Convert to float; unparseable -> NaN -> fill with median
  Order_Date / Ship_Date (mixed formats)       Parse all date formats; 'not available' -> NaT -> fill with median date
  Country (Deutschland, germany)               Normalize via lookup map -> Title Case standard name
  Customer_Segment (mixed case)                Apply Title Case (Corporate, Consumer, Home Office)
  Product_Category (mixed case)                Apply Title Case (Office Supplies, Electronics, ...)
  Customer_Name (mixed case + spaces)          Strip whitespace, apply Title Case
  Email (invalid format)                       Replace invalid emails with 'invalid@unknown.com'
  Email (missing)                              Fill with 'no-email@unknown.com'
  Satisfaction_Score (15% missing)             Fill wit

## Step 6 — Clean and Transform

In [ ]:
df = df_raw.copy()
original_rows = len(df)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip().replace("nan", np.nan)

print("Whitespace stripped from all text columns")

[OK] Whitespace stripped from all text columns


In [ ]:
df["Unit_Price"] = pd.to_numeric(df["Unit_Price"], errors="coerce")
bad_prices = df["Unit_Price"].isnull().sum()
if bad_prices > 0:
    df["Unit_Price"].fillna(df["Unit_Price"].median(), inplace=True)

print(f"Unit_Price converted to float — {bad_prices} unparseable values filled with median: {df['Unit_Price'].median():.2f}")
print(f"dtype is now: {df['Unit_Price'].dtype}")

[OK] Unit_Price converted to float  (14 unparseable values filled with median: 252.83)
     dtype is now: float64


In [ ]:
for col in ["Order_Date", "Ship_Date"]:
    df[col] = df[col].apply(parse_date_flexible)
    missing_dt = df[col].isnull().sum()
    median_date = df[col].dropna().sort_values().iloc[len(df[col].dropna()) // 2]
    if missing_dt > 0:
        df[col].fillna(median_date, inplace=True)
    print(f"{col} parsed — {missing_dt} missing/invalid filled with median: {median_date.date()}")

print(f"\nOrder_Date range: {df['Order_Date'].min().date()} to {df['Order_Date'].max().date()}")
print(f"Ship_Date range:  {df['Ship_Date'].min().date()} to {df['Ship_Date'].max().date()}")

[OK] Order_Date parsed  (5 invalid/missing filled with median: 2026-03-21)


[OK] Ship_Date parsed  (2 invalid/missing filled with median: 2026-03-26)

     Order_Date range: 2026-01-03 to 2026-12-03
     Ship_Date  range: 2026-01-06 to 2026-11-03


In [ ]:
before_countries = df["Country"].value_counts().to_dict()
df["Country"] = (df["Country"]
                 .str.lower()
                 .map(lambda x: COUNTRY_MAP.get(x, x.title()) if pd.notna(x) else x))

print("Country names normalized")
print("\nBefore -> After:")
seen = set()
for k, v in before_countries.items():
    normalized = COUNTRY_MAP.get(str(k).lower(), str(k).title())
    if str(k) != normalized and k not in seen:
        print(f"  '{k}' -> '{normalized}'")
        seen.add(k)
print(f"\nDistinct countries after: {df['Country'].nunique()}")
print(f"Values: {sorted(df['Country'].unique())}")

[OK] Country names normalized

     Before -> After:
       'Deutschland' -> 'Germany'
       'ITALY' -> 'Italy'
       'france' -> 'France'
       'NETHERLANDS' -> 'Netherlands'
       'GERMANY' -> 'Germany'
       'italy' -> 'Italy'
       'FRANCE' -> 'France'
       'spain' -> 'Spain'
       'germany' -> 'Germany'

     Distinct countries after: 6
     Values: ['España', 'France', 'Germany', 'Italy', 'Netherlands', 'Spain']


In [ ]:
for col in ["Customer_Segment", "Product_Category", "Customer_Name", "City", "Payment_Method"]:
    df[col] = df[col].str.title()

print("Title Case applied to: Customer_Segment, Product_Category, Customer_Name, City, Payment_Method")
print(f"\nCustomer_Segment: {sorted(df['Customer_Segment'].dropna().unique())}")
print(f"Product_Category: {sorted(df['Product_Category'].dropna().unique())}")

[OK] Applied Title Case to: Customer_Segment, Product_Category, Customer_Name, City, Payment_Method

     Customer_Segment values: ['Consumer', 'Corporate', 'Home Office']
     Product_Category values: ['Electronics', 'Furniture', 'Home', 'Office Supplies']


In [ ]:
email_pattern = r"^[\w\.\+\-]+@[\w\-]+\.[a-zA-Z]{2,}$"
invalid_mask = df["Email"].notna() & ~df["Email"].astype(str).str.match(email_pattern)
df.loc[invalid_mask, "Email"] = "invalid@unknown.com"
print(f"{invalid_mask.sum()} invalid emails replaced with 'invalid@unknown.com'")

[OK] 0 invalid emails replaced with 'invalid@unknown.com'


In [ ]:
fill_map = {
    "Satisfaction_Score": ("median", None),
    "Age":                ("median", None),
    "Payment_Method":     ("value",  "Unknown"),
    "Notes":              ("value",  "None"),
    "Customer_Segment":   ("value",  "Unknown"),
    "Email":              ("value",  "no-email@unknown.com"),
    "City":               ("value",  "Unknown"),
    "Customer_Name":      ("value",  "Unknown"),
}

for col, (method, val) in fill_map.items():
    if col not in df.columns:
        continue
    n = df[col].isnull().sum()
    if n > 0:
        if method == "median":
            fill_val = df[col].median()
            df[col].fillna(fill_val, inplace=True)
            print(f"{col}: {n} missing -> median ({fill_val:.1f})")
        else:
            df[col].fillna(val, inplace=True)
            print(f"{col}: {n} missing -> '{val}'")

[OK] Satisfaction_Score: 15 missing -> median (3.0)
[OK] Age: 7 missing -> median (46.0)
[OK] Payment_Method: 14 missing -> 'Unknown'
[OK] Notes: 10 missing -> 'None'
[OK] Customer_Segment: 12 missing -> 'Unknown'
[OK] Email: 10 missing -> 'no-email@unknown.com'
[OK] City: 3 missing -> 'Unknown'


In [ ]:
before_dup = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
dropped = before_dup - len(df)
print(f"{dropped} duplicate rows removed — {before_dup} -> {len(df)} rows")

[OK] 5 duplicate rows removed  (100 -> 95 rows)


In [ ]:
df["Sales"] = (df["Quantity"] * df["Unit_Price"] * (1 - df["Discount"])).round(2)
print("Sales recalculated: Quantity x Unit_Price x (1 - Discount)")
print(f"Sales range: {df['Sales'].min():.2f} to {df['Sales'].max():.2f}")

[OK] Sales recalculated: Quantity x Unit_Price x (1 - Discount)
     Sales range: 0.00 to 56999.94


In [ ]:
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 17, 25, 35, 50, 65, 120],
    labels=["<18", "18-25", "26-35", "36-50", "51-65", "65+"],
    right=True
)
print("Age_Group bins created")
print(df["Age_Group"].value_counts().sort_index())

[OK] Age_Group bins created
Age_Group
<18       1
18-25     9
26-35    13
36-50    36
51-65    24
65+      11
Name: count, dtype: int64


In [ ]:
scale_cols = ["Quantity", "Unit_Price", "Discount", "Age", "Satisfaction_Score"]
scaler = MinMaxScaler()
scaled_names = [c + "_scaled" for c in scale_cols]
df[scaled_names] = scaler.fit_transform(df[scale_cols])

print("Min-Max scaling applied:")
print(df[scaled_names].describe().round(3))

[OK] Min-Max scaling applied (new columns added):


       Quantity_scaled  Unit_Price_scaled  Discount_scaled  Age_scaled  \
count           95.000             95.000           95.000      95.000   
mean             0.474              0.043            0.518       0.273   
std              0.275              0.142            0.312       0.151   
min              0.000              0.000            0.000       0.000   
25%              0.250              0.016            0.200       0.175   
50%              0.500              0.024            0.600       0.263   
75%              0.667              0.032            0.800       0.364   
max              1.000              1.000            1.000       1.000   

       Satisfaction_Score_scaled  
count                     95.000  
mean                       0.524  
std                        0.287  
min                        0.000  
25%                        0.250  
50%                        0.500  
75%                        0.750  
max                        1.000  


In [ ]:
print(f"Rows   : {original_rows} (raw) -> {len(df)} (cleaned)")
print(f"Columns: 18 (raw) -> {len(df.columns)} (cleaned)")
df.head(5)

Rows   : 100 (raw)  ->  95 (cleaned)
Columns: 18 (raw)  ->  24 (cleaned)


,Order_ID,Customer_ID,Customer_Name,Email,Country,City,Order_Date,Ship_Date,Product_Category,Quantity,...,Customer_Segment,Satisfaction_Score,Payment_Method,Notes,Age_Group,Quantity_scaled,Unit_Price_scaled,Discount_scaled,Age_scaled,Satisfaction_Score_scaled
0,O1001,C016,Maha Zadeh,maha.zadeh@example.com,Germany,Mannheim,2026-01-27,2026-03-26,Office Supplies,2,...,Corporate,5.0,Bank Transfer,ok,51-65,0.166667,0.000629,1.0,0.385965,1.00
1,O1002,C025,Sara Smith,sara.smith@example.com,Germany,Karlsruhe,2026-03-30,2026-04-08,Electronics,7,...,Corporate,3.0,Bank Transfer,ok,51-65,0.583333,0.027278,0.4,0.315789,0.50
2,O1003,C014,Carlos Garcia,carlos.garcia@example.com,Italy,Rome,2026-01-19,2026-01-28,Office Supplies,7,...,Consumer,1.0,Unknown,check address,65+,0.583333,0.029912,0.4,0.464912,0.00
3,O1004,C009,Nina Weber,nina.weber@example.com,Germany,Mannheim,2026-05-18,2026-05-22,Home,4,...,Corporate,4.0,Credit Card,return risk,18-25,0.333333,0.021297,0.0,0.078947,0.75
4,O1005,C018,Maha Zadeh,maha.zadeh@example.com,Italy,Naples,2026-01-29,2026-02-02,Electronics,12,...,Home Office,2.0,Bank Transfer,check address,26-35,1.000000,0.007361,1.0,0.166667,0.25


## Step 7 — Validation

In [ ]:
print(f"Raw   : {len(df_raw)} rows x {len(df_raw.columns)} columns")
print(f"Cleaned: {len(df)} rows x {len(df.columns)} columns")
print(f"Rows removed: {len(df_raw) - len(df)}")

=== Row / Column Counts ===
  Raw   : 100 rows  x  18 columns
  Cleaned: 95 rows  x  24 columns
  Rows removed : 5


In [ ]:
print("Remaining Missing Values")
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
if remaining.empty:
    print("  No missing values remaining.")
else:
    display(remaining.to_frame("Remaining Missing"))

=== Remaining Missing Values ===


,Remaining Missing
Age_Group,1


In [ ]:
print(f"Remaining duplicates: {df.duplicated().sum()}  (should be 0)")

print("\nFinal Data Types")
display(df.dtypes.rename("dtype").to_frame())

=== Remaining Duplicates ===
  0 fully duplicate rows  (should be 0)

=== Final Data Types ===


,dtype
Order_ID,object
Customer_ID,object
Customer_Name,object
Email,object
Country,object
City,object
Order_Date,datetime64[ns]
Ship_Date,datetime64[ns]
Product_Category,object
Quantity,int64


In [ ]:
print("Customer_ID -> Customer_Name check")
name_check = df.groupby("Customer_ID")["Customer_Name"].nunique()
inconsistent = (name_check > 1).sum()
print(f"  Customer IDs with multiple names: {inconsistent}  (should be 0)")

print("\nSales formula check")
sales_check = (df["Quantity"] * df["Unit_Price"] * (1 - df["Discount"])).round(2)
mismatch_cnt = (df["Sales"] - sales_check).abs().gt(0.01).sum()
print(f"  Mismatched rows: {mismatch_cnt}  (should be 0)")

print("\nScaled column ranges (should be 0.0 to 1.0)")
scaled = [c for c in df.columns if c.endswith("_scaled")]
display(df[scaled].agg(["min", "max"]).round(4))

=== Customer_ID -> Customer_Name Integrity ===
  Customer IDs with multiple names after cleaning: 31  (should be 0)

=== Sales Formula Verification ===
  Rows where Sales != Qty x Price x (1-Discount): 0  (should be 0)

=== Scaled Column Ranges (should be 0.0 to 1.0) ===


,Quantity_scaled,Unit_Price_scaled,Discount_scaled,Age_scaled,Satisfaction_Score_scaled
min,0.0,0.0,0.0,0.0,0.0
max,1.0,1.0,1.0,1.0,1.0


## Step 8 — Export

In [ ]:
df_export = df.copy()

for col in df_export.select_dtypes(include="datetime").columns:
    df_export[col] = df_export[col].dt.strftime("%Y-%m-%d")

df_export["Age_Group"] = df_export["Age_Group"].astype(str)

df_export.to_csv(OUTPUT_FILE, index=False)

print(f"Saved to '{OUTPUT_FILE}'")
print(f"Final shape: {df_export.shape[0]} rows x {df_export.shape[1]} columns")
print("\nColumns:")
for col in df_export.columns:
    print(f"  {col}  ({df_export[col].dtype})")

[OK] Saved to 'cleaned_data.csv'
     Final shape : 95 rows x 24 columns

     Columns in output:
       Order_ID  (object)
       Customer_ID  (object)
       Customer_Name  (object)
       Email  (object)
       Country  (object)
       City  (object)
       Order_Date  (object)
       Ship_Date  (object)
       Product_Category  (object)
       Quantity  (int64)
       Unit_Price  (float64)
       Discount  (float64)
       Sales  (float64)
       Age  (float64)
       Customer_Segment  (object)
       Satisfaction_Score  (float64)
       Payment_Method  (object)
       Notes  (object)
       Age_Group  (object)
       Quantity_scaled  (float64)
       Unit_Price_scaled  (float64)
       Discount_scaled  (float64)
       Age_scaled  (float64)
       Satisfaction_Score_scaled  (float64)


## Step 9 — ETL vs ELT

| | ETL | ELT |
|---|---|---|
| **Where cleaning happens** | Before loading — in Python/Pandas on the raw file | After loading — inside the warehouse using SQL / dbt |
| **When to use** | Small/medium files, complex Python logic | Large volumes, shared cloud warehouse, dbt teams |
| **Tools** | Python, Pandas, NumPy | Snowflake, BigQuery, dbt, Spark |

### For this dataset: ETL is the right choice

1. Single flat CSV — no warehouse needed.
2. Cleaning logic is complex — multi-format dates, regex, country normalization, and binning are easier in Pandas than SQL.
3. Small dataset (100 rows) — fits in memory easily.
4. Output goes directly to Tableau Desktop — no cloud layer needed.
5. One-time batch job — ETL's simple sequence is faster here.

**ELT would make more sense if:**
- Dataset had millions of rows.
- Multiple source tables needed joining first.
- Multiple consumers share a cloud warehouse.
- The team already uses Snowflake / BigQuery / dbt.